# PKG — Counterparty Locatability Harness
### Payment Knowledge Graph · Treasury Management · Data Science
**CPU only · PySpark → pandas · customer↔customer edges as a labelled stand-in for counterparties**

---

## What this notebook is for

We want to infer, for an external counterparty node, where it is and how confident
we are — from a handful of observed edges to PNC customers. The hard case is
**degree 1 or 2**, which is most of the population.

There is no ground truth for counterparties. There is for **customers**: both ends
of a customer↔customer edge are located. So we mask a customer's coordinates,
predict them from *k* of its neighbours, and measure the error. That maps the
achievable-accuracy surface before a single counterparty is scored.

## The reframe that drives the design

At degree 1 the achievable error is not a number, it is **bimodal**. A neighbour
that is a neighbourhood restaurant with a 4 km cloud pins the target inside ~10 km.
A neighbour that is a payment processor leaves the national prior. No estimator
repairs the second case; none is needed for the first.

So the deliverable is **not a better centroid — it is a calibrated gate saying which
nodes are locatable at all, and at what radius.** Confidence is the primary output.
If the answer is "18% of degree-1 counterparties are locatable to a CBSA", that is a
real asset, provided the other 82% are marked unknown rather than given a
plausible-looking point.

## Design decisions

| Decision | Rationale |
|---|---|
| **Degree-matched subsampling**, not stratification | Customers have a very different degree distribution from counterparties. Drawing *k* neighbours at random from a well-connected target yields many k=1 replicates with known truth, plus within-target variance at fixed *k* |
| **Nested draws** — rank once, take prefixes | One random ordering gives every *k* via cumulative sums. Makes error-vs-*k* a *paired* comparison and costs one window instead of one per *k* |
| Each neighbour is a **kernel, not a point** | Centre = its counterparty centroid, bandwidth = its own spread. A hub gets a near-flat kernel and self-cancels — no exclusion list needed |
| **Leave-one-out** neighbour statistics | If *j*'s spread was computed over a set containing the target, using it to predict the target is circular. Exact closed form in §2 |
| **Uniform**, not amount, weighting *inside* the bandwidth | The target is one *counterparty*, not one *dollar*. Block F weights by amount because it describes an economic footprint — a different question |
| Combine by **inverse variance** | Amount is close to anti-correlated with informativeness: the biggest edge is often the processor. Tested against amount and uniform rather than assumed |
| **V0 first** | Hubs are in scope deliberately. Some are government accounts tightly bound to the地 they serve; the question is which high-degree neighbours inform |

## Two switches, and what each isolates

- **`WINDOW`** — `LAST3` or `ALL`. Registered coordinates are *current state* applied
  to every historical month, so a customer that moved contaminates older edges.
  `ALL` buys more neighbours but adds that staleness. **Comparing the two at matched
  *k*** separates the staleness penalty from the extra-data benefit — the only
  measurement of location drift available to us.
- **`VERSION`** — `V0` (hubs present) or `P99_9` (de-hubbed). The difference *is* the
  hub contribution.

---
## 0. Setup

In [ ]:
import os, re, math, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"
PALETTE = px.colors.qualitative.Safe
pd.set_option("display.width", 200, "display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# ---------------------------------------------------------------- config --
EDGE_GLOB    = "/user/pk36814/data/cust_*.csv"     # source,dest,amount,volume
EDGE_FORMAT  = "csv"                                # "csv" | "parquet"
NODE_SOURCE  = "table"                              # "table" | "parquet"
HIVE_TABLE   = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
NODE_PARQUET = "/user/pk36814/metrics/node/*.parquet"

WINDOW    = "LAST3"     # "LAST3" | "ALL"  -> edge aggregation window
VERSION   = "V0"        # "V0" | "P99_9"   -> hubs in / out
REF_MONTH = None        # None -> latest month present

K_GRID         = [1, 2, 3, 5, 8, 12, 20]
K_MAX          = max(K_GRID)
N_REPS         = 5           # random draws per target
MAX_CANDIDATES = 60          # cap neighbours entering the draw (>= 3x K_MAX)
SEED           = 20260812

SHRINK_N0   = 5.0            # pseudo-counts pulling a thin spread to its peer prior
H_FLOOR_KM  = 5.0            # bandwidth floor
HIT_RADII   = [25, 50, 100, 250]

OUT = "../metrics/locatability"
os.makedirs(OUT, exist_ok=True)
TAG = f"{VERSION}_{WINDOW}"
print(f"WINDOW={WINDOW}  VERSION={VERSION}  K_GRID={K_GRID}  N_REPS={N_REPS}  tag={TAG}")

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = (SparkSession.builder
         .appName("pkg_locatability")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.shuffle.partitions", "600")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

R_EARTH_KM = 6371.0088

def to_pd(sdf, label="", max_rows=4_000_000):
    t = time.time(); n = sdf.count()
    if n > max_rows:
        raise MemoryError(f"[{label}] {n:,} rows > max_rows={max_rows:,} — "
                          f"aggregate further in Spark before collecting.")
    df = sdf.toPandas()
    print(f"[{label}] {n:,} rows x {df.shape[1]} cols | {time.time()-t:,.1f}s")
    return df

def xyz(lat, lon):
    """Unit vectors. Every centroid below is computed in 3D and projected
    back: averaging degrees breaks at the date line and distorts with
    latitude."""
    la, lo = F.radians(lat), F.radians(lon)
    return F.cos(la) * F.cos(lo), F.cos(la) * F.sin(lo), F.sin(la)

def hav_km(lat1, lon1, lat2, lon2):
    p1, p2 = F.radians(lat1), F.radians(lat2)
    dp, dl = p2 - p1, F.radians(lon2 - lon1)
    a = F.sin(dp / 2) ** 2 + F.cos(p1) * F.cos(p2) * F.sin(dl / 2) ** 2
    return F.lit(2 * R_EARTH_KM) * F.asin(F.sqrt(F.least(a, F.lit(1.0))))

def hav_km_np(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin((p2-p1)/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def qbucket(sdf, col, n=10, name=None, rel_err=0.001):
    """Decile-style bucketing WITHOUT an unpartitioned window.

    F.ntile(n).over(Window.orderBy(col)) has no partition key, so Spark moves
    the entire frame to one executor to compute a global ordering. Harmless on
    a dry run, fatal on 100M+ rows. approxQuantile is a distributed sketch;
    the cut points come back to the driver and the bucket is a plain column
    expression.
    """
    name = name or f"{col}_bucket"
    qs = sdf.approxQuantile(col, [i / n for i in range(1, n)], rel_err)
    qs = sorted(set(q for q in qs if q is not None))
    e = F.lit(len(qs) + 1)
    for i, q in enumerate(reversed(qs)):
        e = F.when(F.col(col) <= F.lit(float(q)), F.lit(len(qs) - i)).otherwise(e)
    return sdf.withColumn(name, e.cast("int"))


def rbar_to_km(rbar):
    """Mean resultant length -> rms angular spread in km.

    For unit vectors with mean |m| = Rbar, the mean squared CHORD distance to
    the centroid is 1 - Rbar^2. Convert chord to arc. This is the same
    quantity Block F reports as geo_R / geo_spread_km, so the leave-one-out
    values below sit on the manifest's scale.
    """
    chord = F.sqrt(F.greatest(F.lit(0.0), F.lit(1.0) - rbar * rbar))
    return F.lit(2 * R_EARTH_KM) * F.asin(F.least(chord / 2, F.lit(1.0)))

---
## 1. Inputs

Node attributes come from the metrics table at `REF_MONTH` — coordinates, typing,
NAICS and network metrics are all already there, so no dimension join is needed.

The **hub set is derived, not configured**: nodes present at `V0` but absent at
`P99_9` are exactly the ladder's degree/strength exclusions.

In [ ]:
NODES_ALL = (spark.table(HIVE_TABLE) if NODE_SOURCE == "table"
             else spark.read.parquet(NODE_PARQUET))
NODES_ALL = NODES_ALL.toDF(*[c.lower() for c in NODES_ALL.columns])
cols = set(NODES_ALL.columns)

months = sorted(r[0] for r in NODES_ALL.select("time_key").distinct().collect())
REF_MONTH = REF_MONTH or months[-1]
WINDOW_MONTHS = months[-3:] if WINDOW == "LAST3" else months
print(f"{len(months)} months {months[0]}..{months[-1]} | REF_MONTH={REF_MONTH}")
print(f"WINDOW={WINDOW} -> {len(WINDOW_MONTHS)} months "
      f"{WINDOW_MONTHS[0]}..{WINDOW_MONTHS[-1]}")

KEEP = [c for c in
        ["node", "lat", "lon", "zip3", "state", "geo_status", "node_type",
         "entity_type", "naics2", "naics_desc", "cust_name",
         "in_degree", "out_degree", "in_strength", "out_strength",
         "geo_spread_km", "geo_reach_p50_km", "geo_registered_vs_flow_km",
         "geo_locality_class", "pagerank_logw", "clustering_coef", "top_share",
         "share_in_amt_individual", "months_active"] if c in cols]

DIM = (NODES_ALL
       .filter((F.col("time_key") == REF_MONTH) & (F.col("version") == "V0"))
       .select(*KEEP)
       .withColumn("deg_tot", F.coalesce("in_degree", F.lit(0))
                            + F.coalesce("out_degree", F.lit(0)))
       .withColumn("strength", F.coalesce("in_strength", F.lit(0.0))
                             + F.coalesce("out_strength", F.lit(0.0)))
       .withColumn("located", (F.col("geo_status") == "valid").cast("int")))

kept = (NODES_ALL.filter((F.col("time_key") == REF_MONTH)
                         & (F.col("version") == "P99_9"))
        .select("node").distinct().withColumn("in_p999", F.lit(1)))
DIM = (DIM.join(kept, "node", "left")
          .withColumn("is_hub", F.col("in_p999").isNull().cast("int"))
          .drop("in_p999")).cache()

print(f"nodes at {REF_MONTH} (V0): {DIM.count():,}")
DIM.groupBy("is_hub").agg(F.count("*").alias("n"),
                          F.round(F.avg("located"), 4).alias("pct_located"),
                          F.round(F.avg("deg_tot"), 1).alias("avg_degree")).show()

In [ ]:
# ---- edges over the chosen window ---------------------------------------
_fs   = (spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem
         .get(spark.sparkContext._jsc.hadoopConfiguration()))
_Path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path
all_paths = [s.getPath().toString() for s in _fs.globStatus(_Path(EDGE_GLOB))]

def month_of(p):
    m = re.search(r"(\d{4}-\d{2})", os.path.basename(p))
    return m.group(1) if m else None

paths = [p for p in all_paths if month_of(p) in set(WINDOW_MONTHS)]
if not paths:
    raise FileNotFoundError(f"no snapshots matched {EDGE_GLOB} in the window")
print(f"{len(paths)} of {len(all_paths)} snapshot files in window")

if EDGE_FORMAT == "csv":
    SCH = StructType([StructField("source", StringType()),
                      StructField("dest",   StringType()),
                      StructField("amount", DoubleType()),
                      StructField("volume", DoubleType())])
    E = spark.read.csv(paths, header=True, schema=SCH)
else:
    E = spark.read.parquet(*paths)

E = (E.withColumn("source", F.col("source").cast("string"))
       .withColumn("dest",   F.col("dest").cast("string"))
       .filter(F.col("source").isNotNull() & F.col("dest").isNotNull()
               & (F.col("source") != F.col("dest"))))
print(f"raw edge rows in window: {E.count():,}")

### 1.1 The neighbour table

Edges are pooled across the window and symmetrised into one row per
(target, neighbour) carrying both directions.

Pooling is right for *spatial* inference — more edges, same spatial content — but
**months are not independent evidence about location.** A neighbour seen 23 times
is one spatial constraint, not 23. `n_months` is therefore kept as a
*relationship-reality* feature for the confidence model and is never a weight in
the location estimate.

At `VERSION='P99_9'` hubs are dropped from **both** sides, which is what makes the
V0−P99_9 difference readable as the hub contribution.

In [ ]:
agg = (E.groupBy("source", "dest")
        .agg(F.sum("amount").alias("amt"), F.sum("volume").alias("vol"),
             F.count("*").alias("n_months")))

fwd = agg.select(F.col("source").alias("target"), F.col("dest").alias("nb"),
                 F.col("amt").alias("amt_out"), F.col("n_months").alias("mo_out"))
bwd = agg.select(F.col("dest").alias("target"), F.col("source").alias("nb"),
                 F.col("amt").alias("amt_in"), F.col("n_months").alias("mo_in"))

PAIR = (fwd.join(bwd, ["target", "nb"], "outer")
        .fillna(0.0, ["amt_out", "amt_in"]).fillna(0, ["mo_out", "mo_in"])
        .withColumn("amt", F.col("amt_in") + F.col("amt_out"))
        .withColumn("n_months", F.greatest("mo_in", "mo_out"))
        .withColumn("direction",
                    F.when((F.col("amt_in") > 0) & (F.col("amt_out") > 0), "both")
                     .when(F.col("amt_in") > 0, "in_only").otherwise("out_only")))

if VERSION == "P99_9":
    hubs = DIM.filter(F.col("is_hub") == 1).select("node")
    PAIR = (PAIR.join(hubs.withColumnRenamed("node", "target"), "target", "left_anti")
                .join(hubs.withColumnRenamed("node", "nb"), "nb", "left_anti"))
    print("P99_9: hub nodes removed from both sides")

PAIR = PAIR.cache()
print(f"(target, neighbour) pairs: {PAIR.count():,}")

---
## 2. Neighbour kernels and the leave-one-out correction

For neighbour *j* the kernel centre is *j*'s **counterparty centroid** — the mean
position of *j*'s own located counterparties — not *j*'s registered pin. The target
is one of *j*'s counterparties, so the counterparty cloud is the relevant
distribution. (§6 tests centroid against registered pin head to head.)

Everything comes from two sufficient statistics per *j* — the count and `Σu` over
3D unit vectors — so the **leave-one-out values are exact and free**:

```
n' = n − 1        S' = S − u_target
R̄' = |S'| / n'    centroid' = S'/|S'|      spread' = f(R̄')
```

Weighting *inside* the cloud is **uniform, not amount-weighted**: we are asking
"where does a randomly chosen counterparty of *j* sit", and that must not be
dominated by *j*'s single largest relationship.

In [ ]:
# j's cloud is the set of TARGETS attached to j; we need their coordinates.
tpos = DIM.select(F.col("node").alias("target"), F.col("lat").alias("t_lat"),
                  F.col("lon").alias("t_lon"), F.col("located").alias("t_loc"),
                  F.col("state").alias("t_state"))
cx, cy, cz = xyz(F.col("t_lat"), F.col("t_lon"))

CLOUD = (PAIR.select("target", "nb").join(tpos, "target", "inner")
         .filter(F.col("t_loc") == 1)
         .select(F.col("nb").alias("j"), "target",
                 cx.alias("cx"), cy.alias("cy"), cz.alias("cz")))

JS = CLOUD.groupBy("j").agg(F.count("*").alias("j_n"),
                            F.sum("cx").alias("j_sx"),
                            F.sum("cy").alias("j_sy"),
                            F.sum("cz").alias("j_sz"))

LOO = (CLOUD.join(JS, "j", "inner")
       .withColumn("n1", F.col("j_n") - F.lit(1))
       .withColumn("sx", F.col("j_sx") - F.col("cx"))
       .withColumn("sy", F.col("j_sy") - F.col("cy"))
       .withColumn("sz", F.col("j_sz") - F.col("cz"))
       .withColumn("norm", F.sqrt(F.col("sx")**2 + F.col("sy")**2 + F.col("sz")**2))
       .filter((F.col("n1") >= 1) & (F.col("norm") > 0))
       .withColumn("rbar", F.least(F.col("norm") / F.col("n1"), F.lit(1.0)))
       .withColumn("loo_spread_km", rbar_to_km(F.col("rbar")))
       .withColumn("loo_lat", F.degrees(F.asin(F.col("sz") / F.col("norm"))))
       .withColumn("loo_lon", F.degrees(F.atan2(F.col("sy"), F.col("sx"))))
       .select(F.col("j").alias("nb"), "target",
               F.col("n1").alias("nb_n_cp_loo"),
               "loo_lat", "loo_lon", "loo_spread_km"))
print(f"LOO rows (target, neighbour with a usable cloud): {LOO.count():,}")

In [ ]:
nb_attr = DIM.select(
    F.col("node").alias("nb"),
    F.col("lat").alias("nb_lat"), F.col("lon").alias("nb_lon"),
    F.col("state").alias("nb_state"), F.col("naics2").alias("nb_naics2"),
    F.col("node_type").alias("nb_node_type"),
    F.col("entity_type").alias("nb_entity_type"),
    F.col("is_hub").alias("nb_is_hub"), F.col("deg_tot").alias("nb_deg"),
    F.col("strength").alias("nb_strength"), F.col("located").alias("nb_located"),
    *[F.col(c).alias("nb_" + c) for c in
      ["geo_spread_km", "geo_reach_p50_km", "geo_locality_class"]
      if c in DIM.columns])

NB = (LOO.join(PAIR.select("target", "nb", "amt", "n_months", "direction"),
               ["target", "nb"], "inner")
        .join(nb_attr, "nb", "inner")
        .filter(F.col("nb_located") == 1))

# ---- shrink a thin neighbour's spread toward its peer prior --------------
# Most neighbours have too few located counterparties for their own spread to
# mean anything. The prior is naics2 x node_type -- which is where NAICS
# enters the model: not as a direct feature but as the PRIOR ON INTERACTION
# DISTANCE, which is the only channel through which sector can act.
prior = (NB.filter(F.col("nb_n_cp_loo") >= 5)
         .groupBy("nb_naics2", "nb_node_type")
         .agg(F.expr("percentile_approx(loo_spread_km, 0.5)").alias("prior_km"),
              F.count("*").alias("prior_n"))
         .filter(F.col("prior_n") >= 200))
GPRIOR = float(NB.selectExpr(
    "percentile_approx(loo_spread_km, 0.5) g").collect()[0][0])
print(f"global prior spread {GPRIOR:,.1f} km | peer groups {prior.count():,}")

NB = (NB.join(F.broadcast(prior), ["nb_naics2", "nb_node_type"], "left")
      .withColumn("prior_km", F.coalesce("prior_km", F.lit(GPRIOR)))
      .withColumn("h_km", F.sqrt(
          (F.col("nb_n_cp_loo") * F.pow("loo_spread_km", 2)
           + F.lit(SHRINK_N0) * F.pow("prior_km", 2))
          / (F.col("nb_n_cp_loo") + F.lit(SHRINK_N0))))
      .withColumn("h_km", F.greatest("h_km", F.lit(H_FLOOR_KM)))
      .withColumn("w_ivar", F.lit(1.0) / F.pow("h_km", 2))
      .withColumn("w_amt",  F.log1p(F.col("amt")))
      .withColumn("w_unif", F.lit(1.0))).cache()
print(f"neighbour rows with kernels: {NB.count():,}")
NB.select("nb_n_cp_loo", "loo_spread_km", "prior_km", "h_km").summary(
    "min", "25%", "50%", "75%", "max").show()

### 2.1 Who is evaluable, and what that costs

A target enters the harness only if it is itself located (we need truth) and has
at least one located neighbour with a usable cloud. Report the attrition — it is
the denominator for everything downstream, and the counterparty population will
be *worse* on every one of these axes.

In [ ]:
tgt = DIM.select(F.col("node").alias("target"), F.col("lat").alias("y_lat"),
                 F.col("lon").alias("y_lon"), F.col("located").alias("t_located"),
                 F.col("state").alias("t_state"), F.col("naics2").alias("t_naics2"),
                 F.col("node_type").alias("t_node_type"),
                 F.col("entity_type").alias("t_entity_type"),
                 F.col("deg_tot").alias("t_deg"), F.col("strength").alias("t_strength"),
                 F.col("is_hub").alias("t_is_hub"))

EVAL = NB.join(tgt, "target", "inner").filter(F.col("t_located") == 1).cache()
tstat = (EVAL.groupBy("target").agg(F.count("*").alias("n_nb_located")))
print(f"evaluable targets: {tstat.count():,} | "
      f"(target, neighbour) rows: {EVAL.count():,}")

funnel = pd.DataFrame([
    {"stage": "nodes at ref month (V0)", "n": DIM.count()},
    {"stage": "located", "n": DIM.filter(F.col("located") == 1).count()},
    {"stage": "has >=1 located neighbour w/ cloud", "n": tstat.count()},
    {"stage": "has >=2", "n": tstat.filter(F.col("n_nb_located") >= 2).count()},
    {"stage": "has >=5", "n": tstat.filter(F.col("n_nb_located") >= 5).count()},
    {"stage": "has >=20", "n": tstat.filter(F.col("n_nb_located") >= 20).count()}])
funnel["pct_of_nodes"] = funnel.n / funnel.n.iloc[0]
print(funnel.to_string(index=False))
print("\nThe counterparty population will be worse on every line here: fewer "
      "observed edges, and neighbours whose own clouds are thinner.")

---
## 3. The harness

For each target we draw a random ordering of its neighbours **once**, then read
off every *k* as a prefix of that ordering. Cumulative sums along the ordering
give the prediction at every *k* in a single pass, and because the draws are
nested, the error-vs-*k* comparison is **paired within target and replicate** —
the curve is not contaminated by which targets happen to appear at each *k*.

Three weight schemes are carried simultaneously (`ivar`, `amt`, `unif`) and both
centre choices (`loo` centroid, `reg` registered pin), so §6 compares six
estimators on identical draws.

Randomness is a **stable hash**, not `rand()`: Spark may recompute a partition,
and a non-deterministic ordering inside a window silently changes results between
actions.

In [ ]:
lx, ly, lz = xyz(F.col("loo_lat"), F.col("loo_lon"))
rx, ry, rz = xyz(F.col("nb_lat"),  F.col("nb_lon"))
H = (EVAL
     .withColumn("lx", lx).withColumn("ly", ly).withColumn("lz", lz)
     .withColumn("rx", rx).withColumn("ry", ry).withColumn("rz", rz))

# cap candidates per target: keeps the explode bounded on hub-adjacent nodes.
# Cap >= 3 x K_MAX so the draw is still a random subset at every k we report.
ord_all = W.partitionBy("target").orderBy(
    F.xxhash64(F.concat_ws("|", F.col("target"), F.col("nb"), F.lit(SEED))))
H = (H.withColumn("cand_rank", F.row_number().over(ord_all))
       .filter(F.col("cand_rank") <= MAX_CANDIDATES))

H = H.withColumn("rep", F.explode(F.array(*[F.lit(r) for r in range(N_REPS)])))
ordw = W.partitionBy("target", "rep").orderBy(
    F.xxhash64(F.concat_ws("|", F.col("target"), F.col("nb"),
                           F.col("rep"), F.lit(SEED))))
H = H.withColumn("k", F.row_number().over(ordw)).filter(F.col("k") <= K_MAX)

cum = (W.partitionBy("target", "rep").orderBy("k")
        .rowsBetween(W.unboundedPreceding, W.currentRow))
# Selection, not averaging: struct-min carries the coordinates of the
# tightest-bandwidth neighbour seen so far in the prefix. The dry run showed
# hit@50km can FALL with k while the median improves -- averaging pulls toward
# the cloud centre and destroys the case where one tight neighbour nails it.
# This estimator is the counterweight and is the one to watch for precision.
H = (H.withColumn("_tight", F.min(F.struct("h_km", "lx", "ly", "lz")).over(cum))
       .withColumn("s_tight_loo_x", F.col("_tight.lx"))
       .withColumn("s_tight_loo_y", F.col("_tight.ly"))
       .withColumn("s_tight_loo_z", F.col("_tight.lz"))
       .withColumn("h_tight_km", F.col("_tight.h_km")).drop("_tight"))

for wname in ["ivar", "amt", "unif"]:
    w = F.col(f"w_{wname}")
    for cname, (a, b, c) in {"loo": ("lx", "ly", "lz"),
                             "reg": ("rx", "ry", "rz")}.items():
        for ax, src in zip("xyz", (a, b, c)):
            H = H.withColumn(f"s_{wname}_{cname}_{ax}",
                             F.sum(w * F.col(src)).over(cum))
    H = H.withColumn(f"sw_{wname}", F.sum(w).over(cum))
# posterior scale: 1/sqrt(sum of precisions), the analytic concentration
H = (H.withColumn("prec_sum", F.sum(F.lit(1.0) / F.pow("h_km", 2)).over(cum))
       .withColumn("sigma_km", F.lit(1.0) / F.sqrt(F.col("prec_sum")))
       .withColumn("h_min_km", F.min("h_km").over(cum))
       .withColumn("hub_frac", F.avg(F.col("nb_is_hub").cast("double")).over(cum)))

H = H.filter(F.col("k").isin(K_GRID)).cache()
print(f"harness rows (target x rep x k): {H.count():,}")

In [ ]:
def pred_err(df, wname, cname):
    sx, sy, sz = (F.col(f"s_{wname}_{cname}_{a}") for a in "xyz")
    n = F.sqrt(sx*sx + sy*sy + sz*sz)
    return (df.withColumn("p_lat", F.degrees(F.asin(sz / n)))
              .withColumn("p_lon", F.degrees(F.atan2(sy, sx)))
              .withColumn("err_km", hav_km(F.col("p_lat"), F.col("p_lon"),
                                           F.col("y_lat"), F.col("y_lon")))
              .withColumn("scheme", F.lit(f"{wname}_{cname}")))

parts = [pred_err(H, w, c) for w in ["ivar", "amt", "unif"] for c in ["loo", "reg"]]
parts.append(pred_err(H, "tight", "loo"))
keep = ["target", "rep", "k", "scheme", "err_km", "sigma_km", "h_min_km",
        "hub_frac", "t_entity_type", "t_node_type", "t_naics2", "t_state",
        "t_deg", "t_strength"]
PRED = parts[0].select(*keep)
for p in parts[1:]:
    PRED = PRED.unionByName(p.select(*keep))
PRED = PRED.cache()
print(f"prediction rows: {PRED.count():,}")

### 3.1 Baselines

Ordered by how embarrassing it would be to lose to them.

1. **PNC footprint prior** — the population centroid of all located customers.
   A great deal of apparent skill is just "PNC customers are in PA and OH". If the
   estimator does not beat this decisively, it has learned nothing.
2. **Best single neighbour** — an oracle over *k*=1, showing the headroom a
   perfect neighbour-selection rule would buy.
3. **Full-information oracle** — all neighbours, uncapped. The ceiling.

In [ ]:
# 1. footprint prior: 3D mean of every located customer
fp = (DIM.filter(F.col("located") == 1)
      .select(*[f(F.col("lat"), F.col("lon"))[i].alias(a)
                for i, a in enumerate("xyz") for f in [xyz]][:3]
              if False else
              [xyz(F.col("lat"), F.col("lon"))[0].alias("x"),
               xyz(F.col("lat"), F.col("lon"))[1].alias("y"),
               xyz(F.col("lat"), F.col("lon"))[2].alias("z")])
      .agg(F.sum("x").alias("x"), F.sum("y").alias("y"), F.sum("z").alias("z"))
      .collect()[0])
_n = math.sqrt(fp["x"]**2 + fp["y"]**2 + fp["z"]**2)
FP_LAT = math.degrees(math.asin(fp["z"]/_n))
FP_LON = math.degrees(math.atan2(fp["y"], fp["x"]))
print(f"PNC footprint prior: {FP_LAT:.3f}, {FP_LON:.3f}")

base_prior = (tgt.filter(F.col("t_located") == 1)
              .join(tstat, "target", "inner")
              .withColumn("err_km", hav_km(F.lit(FP_LAT), F.lit(FP_LON),
                                           F.col("y_lat"), F.col("y_lon")))
              .select("target", "err_km").withColumn("scheme", F.lit("footprint_prior")))

# 2. best single neighbour (oracle over k=1)
k1 = PRED.filter((F.col("k") == 1) & (F.col("scheme") == "ivar_loo"))
base_best1 = (k1.groupBy("target").agg(F.min("err_km").alias("err_km"))
              .withColumn("scheme", F.lit("best_single_nb_oracle")))

# 3. full-information oracle: all neighbours, uncapped, inverse-variance
allx, ally, allz = xyz(F.col("loo_lat"), F.col("loo_lon"))
ORC = (EVAL.withColumn("x", allx).withColumn("y", ally).withColumn("z", allz)
       .groupBy("target")
       .agg(F.sum(F.col("w_ivar") * F.col("x")).alias("sx"),
            F.sum(F.col("w_ivar") * F.col("y")).alias("sy"),
            F.sum(F.col("w_ivar") * F.col("z")).alias("sz"),
            F.count("*").alias("n_nb"))
       .withColumn("n", F.sqrt(F.col("sx")**2 + F.col("sy")**2 + F.col("sz")**2))
       .withColumn("p_lat", F.degrees(F.asin(F.col("sz")/F.col("n"))))
       .withColumn("p_lon", F.degrees(F.atan2(F.col("sy"), F.col("sx"))))
       .join(tgt.select("target", "y_lat", "y_lon"), "target", "inner")
       .withColumn("err_km", hav_km(F.col("p_lat"), F.col("p_lon"),
                                    F.col("y_lat"), F.col("y_lon"))))
base_orc = ORC.select("target", "err_km").withColumn("scheme", F.lit("oracle_all_nb"))

def qstats(sdf, by=("scheme",)):
    aggs = [F.count("*").alias("n"),
            F.expr("percentile_approx(err_km, 0.5)").alias("p50_km"),
            F.expr("percentile_approx(err_km, 0.9)").alias("p90_km")]
    aggs += [F.avg((F.col("err_km") <= r).cast("double")).alias(f"hit_{r}km")
             for r in HIT_RADII]
    return sdf.groupBy(*by).agg(*aggs)

BASE = to_pd(qstats(base_prior.unionByName(base_best1).unionByName(base_orc)),
             "baselines")
print(BASE.to_string(index=False))

---
## 4. Error versus *k* — the core result

Median and p90, plus hit-rates. **The mean is meaningless here**: the distribution
has a national tail, and one processor-only target moves it more than a thousand
well-located ones.

In [ ]:
# Two curves, and the distinction matters.
#   all      -- every evaluable target. This is what we will actually see, but
#               the population CHANGES with k: only high-degree targets survive
#               to k=20, and those are the least locatable, so the curve can
#               bend the wrong way for a compositional reason.
#   cohort   -- targets with at least K_MAX located neighbours. Same nodes at
#               every k, so this is the genuine "does more evidence help"
#               answer and the only paired one.
rich = tstat.filter(F.col("n_nb_located") >= K_MAX).select("target")
PRED = PRED.join(rich.withColumn("in_cohort", F.lit(1)), "target", "left") \
           .withColumn("in_cohort", F.coalesce("in_cohort", F.lit(0)))

c_all = qstats(PRED, by=("scheme", "k")).withColumn("pop", F.lit("all"))
c_coh = qstats(PRED.filter(F.col("in_cohort") == 1),
               by=("scheme", "k")).withColumn("pop", F.lit("cohort"))
CURVE = to_pd(c_all.unionByName(c_coh).orderBy("pop", "scheme", "k"), "error vs k")
CURVE.to_parquet(f"{OUT}/curve_{TAG}.parquet", index=False)
for pop in ["cohort", "all"]:
    print(f"\n--- p50_km, population = {pop} ---")
    print(CURVE[CURVE["pop"] == pop].pivot(index="k", columns="scheme",
                                           values="p50_km").to_string())

In [ ]:
best = CURVE[(CURVE.scheme == "ivar_loo") & (CURVE["pop"] == "cohort")]
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Median / p90 error vs number of neighbours", "Hit-rate vs k (ivar_loo)"))
for col, dash in [("p50_km", "solid"), ("p90_km", "dot")]:
    fig.add_scatter(x=best.k, y=best[col], name=col, mode="lines+markers",
                    line=dict(dash=dash, width=3), row=1, col=1)
fig.add_hline(y=float(BASE.loc[BASE.scheme == "footprint_prior", "p50_km"].iloc[0]),
              line_dash="dash", line_color="crimson", row=1, col=1,
              annotation_text="footprint prior (p50)")
for i, r in enumerate(HIT_RADII):
    fig.add_scatter(x=best.k, y=best[f"hit_{r}km"], name=f"<= {r} km",
                    mode="lines+markers", line=dict(color=PALETTE[i]), row=1, col=2)
fig.update_yaxes(type="log", title_text="km", row=1, col=1)
fig.update_yaxes(tickformat=".0%", title_text="share of targets", row=1, col=2)
fig.update_xaxes(title_text="k neighbours drawn")
fig.update_layout(height=440, title=f"Locatability vs evidence — {TAG}")
fig.show()

k1 = best[best.k == 1].iloc[0]
print(f"AT k=1: median {k1.p50_km:,.0f} km | p90 {k1.p90_km:,.0f} km | "
      f"within 50 km {k1.hit_50km:.1%} | within 250 km {k1.hit_250km:.1%}")
print("The k=1 hit-rate is the single most important number in this notebook: "
      "it is the share of the high-volume degree-1 counterparty population that "
      "is locatable at all.")

In [ ]:
# Estimator comparison on identical draws.
fig = px.line(CURVE[CURVE["pop"] == "cohort"], x="k", y="p50_km",
              color="scheme", markers=True,
              log_y=True, color_discrete_sequence=PALETTE,
              title="Estimator comparison — identical draws, paired within "
                    "target and replicate")
fig.update_layout(height=420, yaxis_title="median error (km)")
fig.show()
piv = CURVE[CURVE["pop"] == "cohort"].pivot(index="k", columns="scheme",
                                            values="hit_50km")
print("hit@50km by estimator:\n", piv.to_string())
print("\nloo vs reg answers 'centroid or registered pin'. ivar vs amt answers "
      "'is the dollar-weighted edge the informative one' — expect NO, because "
      "the largest edge is often the processor.")

### 4.1 Within-target variance at fixed *k*

How much of the k=1 error is *how many* neighbours you have versus **which one you
happened to get**? This is the quantity that decides whether a selection rule is
worth building: if the spread across replicates within a target is large, then
choosing the right neighbour matters more than finding more of them.

In [ ]:
wt = (PRED.filter((F.col("scheme") == "ivar_loo") & (F.col("k") == 1))
      .groupBy("target").agg(F.count("*").alias("n"),
                             F.min("err_km").alias("best"),
                             F.max("err_km").alias("worst"),
                             F.expr("percentile_approx(err_km,0.5)").alias("med"))
      .filter(F.col("n") >= 3))
wtp = to_pd(wt.sample(False, min(1.0, 200000 / max(wt.count(), 1)), seed=1),
            "within-target spread")
wtp["ratio"] = wtp.worst / wtp.best.clip(lower=1)
print(wtp[["best", "med", "worst", "ratio"]].describe().T.to_string())
fig = px.histogram(wtp[wtp.ratio.between(1, 1000)], x="ratio", nbins=60, log_x=True,
                   title="Worst / best k=1 error within the same target — "
                         "how much does WHICH neighbour matter?")
fig.update_layout(height=360, xaxis_title="ratio (log)")
fig.show()
print(f"median worst/best ratio = {wtp.ratio.median():,.1f}x — if this is large, "
      f"neighbour SELECTION beats neighbour COUNT, and the gate in section 6 is "
      f"the product rather than the estimator.")

---
## 5. What makes a neighbour informative

Every k=1 row is a clean natural experiment: one neighbour, one target, one error.
Stratifying those rows by the neighbour's attributes answers the questions directly
— which sectors locate, whether hubs are uniformly useless, whether amount helps,
and whether the neighbour's own spread is the thing that matters.

In [ ]:
K1 = (H.filter(F.col("k") == 1)
      .join(PRED.filter((F.col("k") == 1) & (F.col("scheme") == "ivar_loo"))
                .select("target", "rep", "err_km"), ["target", "rep"], "inner")
      .select("target", "rep", "err_km", "nb", "nb_naics2", "nb_node_type",
              "nb_entity_type", "nb_is_hub", "nb_deg", "nb_strength",
              "nb_n_cp_loo", "loo_spread_km", "h_km", "amt", "n_months",
              "direction", "nb_state", "t_state", "t_entity_type", "t_deg")
      .withColumn("hit50", (F.col("err_km") <= 50).cast("double"))
      .withColumn("hit250", (F.col("err_km") <= 250).cast("double"))).cache()
print(f"k=1 natural experiments: {K1.count():,}")

def strat(by, min_n=500):
    return (K1.groupBy(*by).agg(
                F.count("*").alias("n"),
                F.expr("percentile_approx(err_km,0.5)").alias("p50_km"),
                F.avg("hit50").alias("hit_50km"),
                F.avg("hit250").alias("hit_250km"),
                F.expr("percentile_approx(loo_spread_km,0.5)").alias("nb_spread_km"),
                F.expr("percentile_approx(nb_deg,0.5)").alias("nb_deg_med"))
            .filter(F.col("n") >= min_n))

In [ ]:
# ---- by neighbour sector ------------------------------------------------
S = to_pd(strat(["nb_naics2"]).orderBy(F.desc("hit_50km")), "by naics2")
S = S[S.nb_naics2.notna()]
fig = px.bar(S, x="nb_naics2", y="hit_50km", color="nb_spread_km",
             hover_data=["n", "p50_km", "nb_deg_med"],
             color_continuous_scale="Tealrose_r",
             title="Which sector of neighbour locates a target? "
                   "hit@50km from ONE neighbour, coloured by that sector's spread")
fig.update_layout(height=430, yaxis_tickformat=".0%",
                  xaxis_title="neighbour naics2", xaxis_type="category")
fig.show()
print(S.to_string(index=False))
print("\nRead this against the spread colouring: if the sector ordering is just "
      "the spread ordering, NAICS adds nothing beyond the interaction-distance "
      "prior already in h_km, and should not be a separate model feature.")

In [ ]:
# ---- the hub question, and the government-account hypothesis ------------
hub = to_pd(strat(["nb_is_hub"]), "hub vs not")
print(hub.to_string(index=False))

# Degree deciles -- is informativeness monotone in degree, or does sector cut
# across it? Government accounts (naics2 = 92) are high-degree but tightly
# bound to the geography they serve; if the hub effect is really a spread
# effect, sector 92 should sit far above other high-degree neighbours.
K1D = qbucket(K1, "nb_deg", 10, "deg_decile")
dd = to_pd(K1D.groupBy("deg_decile").agg(
    F.count("*").alias("n"), F.avg("hit50").alias("hit_50km"),
    F.expr("percentile_approx(nb_deg,0.5)").alias("deg_med"),
    F.expr("percentile_approx(loo_spread_km,0.5)").alias("spread_med")
    ).orderBy("deg_decile"), "degree deciles")
print(dd.to_string(index=False))

top_deg = K1D.filter(F.col("deg_decile") >= 9)
gov = to_pd(top_deg.groupBy("nb_naics2").agg(
    F.count("*").alias("n"), F.avg("hit50").alias("hit_50km"),
    F.expr("percentile_approx(loo_spread_km,0.5)").alias("spread_med")
    ).filter(F.col("n") >= 300).orderBy(F.desc("hit_50km")), "top-decile degree by sector")
fig = px.bar(gov[gov.nb_naics2.notna()], x="nb_naics2", y="hit_50km",
             hover_data=["n", "spread_med"], color_discrete_sequence=PALETTE,
             title="Within the TOP DEGREE DECILE only — hit@50km by sector. "
                   "High degree is not uniformly uninformative")
fig.update_layout(height=400, yaxis_tickformat=".0%", xaxis_type="category")
fig.show()
print(gov.to_string(index=False))
print("\nIf sector 92 (public administration) and other place-bound sectors sit "
      "high here, the correct policy is a hub TAXONOMY -- some hubs are excellent "
      "locators -- not a flat exclusion list.")

In [ ]:
# ---- amount, relationship length, direction, and the neighbour's own spread
K1B = (qbucket(K1, "amt", 10, "amt_decile")
         .withColumn("spread_bucket", F.when(F.col("loo_spread_km") < 25, "<25km")
                     .when(F.col("loo_spread_km") < 100, "25-100km")
                     .when(F.col("loo_spread_km") < 500, "100-500km")
                     .otherwise(">=500km"))
         .withColumn("mo_bucket", F.when(F.col("n_months") <= 1, "1")
                     .when(F.col("n_months") <= 3, "2-3")
                     .when(F.col("n_months") <= 11, "4-11").otherwise("12+")))
panels = {"amt_decile": "edge amount decile", "spread_bucket": "neighbour own spread",
          "mo_bucket": "months of relationship", "direction": "edge direction"}
fig = make_subplots(rows=2, cols=2, subplot_titles=list(panels.values()))
for i, (col, _t) in enumerate(panels.items()):
    d = to_pd(K1B.groupBy(col).agg(F.count("*").alias("n"),
                                   F.avg("hit50").alias("hit_50km")
              ).filter(F.col("n") >= 500).orderBy(col), col)
    fig.add_bar(x=d[col].astype(str), y=d.hit_50km, name=col,
                marker_color=PALETTE[i], row=i // 2 + 1, col=i % 2 + 1)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=620, showlegend=False,
                  title="hit@50km from a single neighbour, by edge and "
                        "neighbour attributes")
fig.show()
print("Amount is the one to watch: if the hit-rate is FLAT or DECREASING in the "
      "amount decile, dollar-weighting the estimator is actively wrong, and the "
      "ivar-vs-amt comparison in section 4 should already agree.")

---
## 6. The gate — predicting locatability before we know the answer

Everything above is descriptive. The product is a model that, **given only what we
can see about a counterparty's neighbours**, predicts whether the estimate will
land inside 50 km. That probability is the confidence output, and the threshold on
it is the locatable/not-locatable decision.

Features are strictly observable for a real counterparty: the neighbour's own
attributes and cloud, the edge, and the posterior scale. **The target's own
attributes are excluded** — we would not have them.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve

FEATS_NUM = ["nb_deg", "nb_strength", "nb_n_cp_loo", "loo_spread_km", "h_km",
             "amt", "n_months"]
FEATS_CAT = ["nb_naics2", "nb_node_type", "direction", "nb_is_hub"]
SAMPLE_N  = 800_000

G = to_pd(K1.select(*FEATS_NUM, *FEATS_CAT, "hit50", "target", "err_km")
            .sample(False, min(1.0, SAMPLE_N / max(K1.count(), 1)), seed=7),
          "gate training frame", max_rows=2_000_000)
for c in FEATS_CAT:
    G[c] = G[c].astype("category")
for c in ["nb_deg", "nb_strength", "amt"]:
    G[c] = np.log1p(G[c].clip(lower=0))

# split on TARGET, not row: replicates of one target must not straddle the split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
itr, ite = next(gss.split(G, groups=G.target))
Xtr, Xte = G.iloc[itr][FEATS_NUM + FEATS_CAT], G.iloc[ite][FEATS_NUM + FEATS_CAT]
ytr, yte = G.iloc[itr].hit50, G.iloc[ite].hit50

clf = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
    categorical_features=[FEATS_NUM.__len__() + i for i in range(len(FEATS_CAT))],
    random_state=0).fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
print(f"base rate hit@50km = {yte.mean():.3f}")
print(f"AUC = {roc_auc_score(yte, p):.4f} | Brier = {brier_score_loss(yte, p):.4f}")

In [ ]:
imp = permutation_importance(clf, Xte.sample(min(60000, len(Xte)), random_state=0),
                             yte.loc[Xte.sample(min(60000, len(Xte)),
                                                random_state=0).index],
                             n_repeats=5, random_state=0, scoring="roc_auc")
I = (pd.DataFrame({"feature": FEATS_NUM + FEATS_CAT,
                   "importance": imp.importances_mean,
                   "sd": imp.importances_std})
     .sort_values("importance"))
fig = px.bar(I, x="importance", y="feature", orientation="h", error_x="sd",
             color_discrete_sequence=PALETTE,
             title="What actually predicts locatability (permutation AUC drop)")
fig.update_layout(height=440)
fig.show()
print(I.sort_values("importance", ascending=False).to_string(index=False))
print("\nIf loo_spread_km / h_km dominate and nb_naics2 is near zero, sector is "
      "fully mediated by interaction distance -- which is the cleaner story and "
      "means the gate transfers to counterparties without a NAICS on them.")

In [ ]:
# ---- calibration and the locatable-fraction table -----------------------
frac, mean_pred = calibration_curve(yte, p, n_bins=15, strategy="quantile")
fig = go.Figure()
fig.add_scatter(x=mean_pred, y=frac, mode="lines+markers", name="observed",
                line=dict(width=3))
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect",
                line=dict(dash="dash", color="grey"))
fig.update_layout(height=420, title="Gate calibration — predicted vs observed "
                  "P(within 50 km)", xaxis_title="predicted", yaxis_title="observed",
                  xaxis_tickformat=".0%", yaxis_tickformat=".0%")
fig.show()

rows = []
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = p >= thr
    if m.sum() < 100: continue
    e = G.iloc[ite].err_km.to_numpy()[m]
    rows.append({"threshold": thr, "share_gated_in": m.mean(),
                 "n": int(m.sum()), "precision_hit50": yte.to_numpy()[m].mean(),
                 "p50_km": np.median(e), "p90_km": np.quantile(e, 0.9)})
GATE = pd.DataFrame(rows)
GATE.to_parquet(f"{OUT}/gate_{TAG}.parquet", index=False)
print(GATE.to_string(index=False))
print("\nTHIS TABLE IS THE DELIVERABLE. Read a row as: 'at this threshold we "
      "publish a location for X% of degree-1 counterparties, and of those, Y% are "
      "within 50 km.' Pick the threshold with the business, not here.")

---
## 7. Contrasts

Three comparisons the notebook is built to support. Each needs a full re-run with
the switch flipped; results are written per `TAG` and read back here.

In [ ]:
# ---- 7.1 out-of-footprint: the closest analogue to a counterparty -------
# A counterparty's observed neighbours are all PNC customers, which biases every
# prediction toward the PNC footprint. Customers registered OUTSIDE the core
# states are the nearest available proxy for that failure mode.
core = to_pd(DIM.filter(F.col("located") == 1).groupBy("state")
             .agg(F.count("*").alias("n")).orderBy(F.desc("n")).limit(8), "core states")
CORE = set(core.state.dropna())
print("core footprint states:", sorted(CORE))

OF = (PRED.filter(F.col("scheme") == "ivar_loo")
      .filter(F.col("in_cohort") == 1)
      .withColumn("in_core", F.col("t_state").isin(list(CORE)))
      .groupBy("in_core", "k").agg(
          F.count("*").alias("n"),
          F.expr("percentile_approx(err_km,0.5)").alias("p50_km"),
          F.avg((F.col("err_km") <= 50).cast("double")).alias("hit_50km")))
ofp = to_pd(OF.orderBy("k", "in_core"), "out-of-footprint")
fig = px.line(ofp, x="k", y="hit_50km", color=ofp.in_core.map(
                {True: "core footprint", False: "outside footprint"}),
              markers=True, log_x=True, color_discrete_sequence=PALETTE,
              title="Locatability inside vs outside the PNC deposit footprint")
fig.update_layout(height=400, yaxis_tickformat=".0%", legend_title=None)
fig.show()
print(ofp.pivot(index="k", columns="in_core", values="hit_50km").to_string())
print("\nThe gap here IS the systematic bias that will hit counterparties, most "
      "of which bank elsewhere. Quote it as a haircut on every headline number.")

In [ ]:
# ---- 7.2 window: staleness vs more data, at MATCHED k -------------------
# Registered coordinates are current-state applied to every month, so ALL adds
# location drift. Comparing at matched k holds evidence volume fixed, which is
# the only way to separate the two effects.
import glob as _g
found = {os.path.basename(f).replace("curve_", "").replace(".parquet", ""): f
         for f in _g.glob(f"{OUT}/curve_*.parquet")}
print("runs available:", sorted(found))
if len(found) > 1:
    C = pd.concat([pd.read_parquet(f).assign(tag=t) for t, f in found.items()])
    C = C[(C.scheme == "ivar_loo") & (C["pop"] == "cohort")]
    fig = px.line(C, x="k", y="hit_50km", color="tag", markers=True, log_x=True,
                  color_discrete_sequence=PALETTE,
                  title="Matched-k comparison across runs — window and version")
    fig.update_layout(height=420, yaxis_tickformat=".0%")
    fig.show()
    print(C.pivot(index="k", columns="tag", values="hit_50km").to_string())
    print("\nLAST3 vs ALL at the SAME k isolates location staleness: same evidence "
          "volume, older edges. If ALL is worse at matched k, that difference is "
          "the drift penalty. If ALL is better, drift is negligible and pooling "
          "wins on neighbour availability alone (compare the funnel in section 2.1).")
    print("V0 vs P99_9 at the same k is the hub contribution.")
else:
    print("Only one run present. Flip WINDOW / VERSION at the top, re-run, and "
          "return here — the comparison reads whatever curve_*.parquet exists.")

---
## 8. Reading the results, and what to build

**The three numbers that decide the programme**

1. `hit_50km` at *k*=1 — the share of the degree-1 counterparty population that is
   locatable at all. Apply the out-of-footprint haircut from §7.1 before quoting it.
2. The **gate table** in §6 — the honest precision/coverage frontier. This is the
   deliverable, not the centroid.
3. The **worst/best ratio** in §4.1 — if large, neighbour *selection* dominates
   neighbour *count*, and the gate is worth more than any estimator refinement.

**Known limits of this harness**

- **Customer-derived curves are optimistic.** Counterparties bank elsewhere, which
  correlates with sitting outside the footprint, and their observed neighbour sets
  are a more biased sample of their true counterparty base. §7.1 bounds the
  direction and rough size; it does not remove the bias.
- **The evaluable set is not the population.** §2.1 shows the attrition. Everything
  here conditions on having a located neighbour with a usable cloud.
- **`n_months` is not independent evidence.** Pooling is right for the spatial
  signal, but a neighbour seen 23 times is one constraint. It is used only as a
  relationship-reality feature in the gate.
- **`MAX_CANDIDATES` truncates very high-degree targets.** Immaterial at the *k*
  values reported, but the oracle in §3.1 is computed uncapped for that reason.

**Sequence from here**

1. Run all four cells of the switch matrix — `V0/LAST3`, `V0/ALL`, `P99_9/LAST3`,
   `P99_9/ALL` — and read §7.2. That settles the pooling question and the hub
   question with one table.
2. If §5 shows a hub taxonomy (place-bound high-degree sectors locating well),
   promote it into the Hub Node Registry as a per-class policy rather than an
   exclusion list.
3. **Get external ground truth before quoting any accuracy figure outside the
   team.** The FI pinning registry — FDIC Summary of Deposits and NCUA — gives
   counterparties with known locations from the *right* population. Small, but it
   measures the customer-to-counterparty gap that §7.1 can only bound.
4. Only then port the estimator to counterparty nodes, and publish locations
   exclusively above the gate threshold, with the radius attached.

**Governance.** Inferring locations for non-customers to support prospecting sits
inside the compliance read already flagged as a precondition for counterparty work.
The estimator here is auditable line by line — kernel centres, bandwidths, weights
are all inspectable — which is a property worth preserving in whatever replaces it.